In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/electra-fulldataset/config.json
/kaggle/input/electra-fulldataset/training_args.bin
/kaggle/input/electra-fulldataset/model.safetensors
/kaggle/input/xlnet-fulldataset/config.json
/kaggle/input/xlnet-fulldataset/training_args.bin
/kaggle/input/xlnet-fulldataset/model.safetensors
/kaggle/input/roberta-fulldataset/config.json
/kaggle/input/roberta-fulldataset/training_args.bin
/kaggle/input/roberta-fulldataset/model.safetensors
/kaggle/input/recheckaspectdataset/udemy_comments_with_aspects_chunk3.csv
/kaggle/input/commentswithaspectchunk/udemy_comments_with_aspects_chunk2.csv
/kaggle/input/commentswithaspectchunk/udemy_comments_with_aspects_chunk1.csv
/kaggle/input/commentswithaspectchunk/udemy_comments_with_aspects_chunk3.csv


In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# กำหนด model paths, base tokenizer names, และ max lengths
model_configs = [
    ("/kaggle/input/electra-fulldataset", "google/electra-base-discriminator", 64),
    ("/kaggle/input/roberta-fulldataset", "roberta-base", 128),
    ("/kaggle/input/xlnet-fulldataset", "xlnet-base-cased", 128)
]

# Load models and tokenizers
models = []
tokenizers = []
max_lengths = []
for model_path, base_name, max_length in model_configs:
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    models.append(model)
    tokenizers.append(AutoTokenizer.from_pretrained(base_name))
    max_lengths.append(max_length)

# Load the first chunk of data
df = pd.read_csv("/kaggle/input/recheckaspectdataset/udemy_comments_with_aspects_chunk3.csv")

# Remove rows where predicted_aspect is Ambiguous
df = df[df['predicted_aspect'] != 'Ambiguous']

def predict(text, model, tokenizer, max_length):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probabilities.squeeze().cpu().numpy()

def ensemble_predict(text, models, tokenizers, max_lengths):
    predictions = [predict(text, model, tokenizer, max_length) 
                   for model, tokenizer, max_length in zip(models, tokenizers, max_lengths)]
    avg_probs = np.mean(predictions, axis=0)
    return np.argmax(avg_probs)

# Perform predictions
predictions = []
for sentence in tqdm(df['sentence'], desc="Predicting sentiments"):
    prediction = ensemble_predict(sentence, models, tokenizers, max_lengths)
    predictions.append(prediction)

# Map numerical predictions to sentiment labels
sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df['predicted_sentiment_ensemble'] = [sentiment_map[pred] for pred in predictions]

# Save results (including all original columns)
df.to_csv('ensemble_predictions_chunk3_full_custom_max_length_no_ambiguous.csv', index=False)
print("Predictions completed and saved to 'ensemble_predictions_chunk3_full_custom_max_length_no_ambiguous.csv'")

# Display first few rows of the results
print(df[['sentence', 'predicted_sentiment_ensemble', 'predicted_aspect'] + [col for col in df.columns if col not in ['sentence', 'predicted_sentiment_ensemble', 'predicted_aspect']]].head())

# Optional: Calculate accuracy if 'predicted_sentiment' column exists
if 'predicted_sentiment' in df.columns:
    accuracy = (df['predicted_sentiment'] == df['predicted_sentiment_ensemble']).mean()
    print(f"Accuracy of ensemble predictions compared to original predictions: {accuracy:.4f}")

# Optional: Display distribution of predictions
print("\nDistribution of ensemble predictions:")
print(df['predicted_sentiment_ensemble'].value_counts(normalize=True))

# Display distribution of aspects after removing Ambiguous
print("\nDistribution of aspects after removing Ambiguous:")
print(df['predicted_aspect'].value_counts(normalize=True))

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (5 outputs left out)

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Predicting sentiments: 100%|██████████| 469933/469933 [4:59:35<00:00, 26.14it/s]


Predictions completed and saved to 'ensemble_predictions_chunk3_full_custom_max_length_no_ambiguous.csv'
                                             sentence  \
3                   positive value for the money paid   
7        thank u for the demo videos those are great.   
17  The practice exam given has answers to the que...   
18  A very good Positive is there are lot of links...   
19  Mr. Arpan you have done a Great Job by explain...   

   predicted_sentiment_ensemble       predicted_aspect  \
3                      Positive              Financial   
7                      Positive      Design & Material   
17                     Negative  Assignment & Tutorial   
18                     Positive  Assignment & Tutorial   
19                     Positive         Skill & Career   

                        group_category  \
3   Personal Development and Lifestyle   
7                    Technology and IT   
17                   Technology and IT   
18                   Technology and